# GENERATE MOCK DATA

---

### Import Libraries

In [35]:
import random
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta

---

### Generate normal transaction

In [36]:
RANDOM_SEED = 42
TARGET_TX   = 200_000

START = datetime(2026, 6, 24, 12, 0, 0, tzinfo=timezone.utc)
END   = datetime(2026, 6, 25, 12, 0, 0, tzinfo=timezone.utc)

_HOUR_WEIGHTS = np.array([
    0.4, 0.2, 0.2, 0.2, 0.3, 0.7,   # 00h–05h: đêm muộn / rạng sáng
    1.5, 3.0, 4.5, 5.5, 5.5, 5.8,   # 06h–11h: sáng sớm → giờ làm việc
    6.0, 5.5, 4.8, 4.2, 4.5, 5.0,   # 12h–17h: trưa đỉnh → chiều
    4.2, 3.5, 3.0, 2.5, 1.5, 0.8,   # 18h–23h: tối → đêm
])
_HOUR_WEIGHTS = _HOUR_WEIGHTS / _HOUR_WEIGHTS.sum()

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [37]:
def sample_n_tx() -> int:
    group = random.choices([0, 1, 2], weights=[35, 50, 15])[0]
    return random.choice([(1, 2, 3), (4, 5, 6), (7, 8, 9)][group])

def sample_avg_amount() -> float:
    tier = random.choices([0, 1, 2], weights=[60, 30, 10])[0]
    lo, hi = [(500_000, 2_000_000), (2_000_000, 15_000_000), (30_000_000, 50_000_000)][tier]
    return random.uniform(lo, hi)

def sample_amount(avg: float) -> int:
    raw     = np.random.normal(avg, avg * 0.15)
    clipped = max(avg * 0.50, min(avg * 3.50, raw))
    return int(max(50_000, round(clipped / 1_000) * 1_000))

def sample_event_time() -> str:
    hour_of_day = int(np.random.choice(24, p=_HOUR_WEIGHTS))
    window_offset_h = (hour_of_day - 12) % 24
    offset_s = window_offset_h * 3600 + random.uniform(0, 3600)
    return (START + timedelta(seconds=offset_s)).isoformat()

In [38]:
all_accounts = [f"ML_{i:05d}" for i in range(100_000)]
random.shuffle(all_accounts)

rows = []
for acc_id in all_accounts:
    avg_amount = sample_avg_amount()
    n_tx       = sample_n_tx()

    for _ in range(n_tx):
        rows.append({
            "account_id":  acc_id,
            "amount":      sample_amount(avg_amount),
            "event_time":  sample_event_time(),
            "is_home":     True,
            "is_domestic": True,
            "is_fraud":    False,
        })

    if len(rows) >= TARGET_TX:
        break

print(f"Transactions : {len(rows):,}")
print(f"Accounts     : {len(set(r['account_id'] for r in rows)):,}")

Transactions : 200,004
Accounts     : 45,550


In [39]:
df_normal_transaction = (
    pd.DataFrame(rows)
      .sort_values("event_time")
      .reset_index(drop=True)
)

print(df_normal_transaction.shape)
df_normal_transaction.head()

(200004, 6)


,account_id,amount,event_time,is_home,is_domestic,is_fraud
0,ML_81908,3116000,2026-06-24T12:00:00.605527+00:00,True,True,False
1,ML_70721,7763000,2026-06-24T12:00:00.719117+00:00,True,True,False
2,ML_33203,951000,2026-06-24T12:00:01.057246+00:00,True,True,False
3,ML_43504,2553000,2026-06-24T12:00:01.433330+00:00,True,True,False
4,ML_08929,3880000,2026-06-24T12:00:01.487840+00:00,True,True,False


---

### Inject Fraud Transaction

In [40]:
FRAUD_SEED = 123
random.seed(FRAUD_SEED)
np.random.seed(FRAUD_SEED)

In [41]:
acc_avg = df_normal_transaction.groupby("account_id")["amount"].mean()
all_normal_accounts = acc_avg.index.tolist()

In [42]:
def round_vnd(x: float) -> int:
    return int(max(50_000, round(x / 1_000) * 1_000))

def fraud_base_time(night_prob: float = 0.5, margin_minutes: int = 30) -> datetime:
    if random.random() < night_prob:
        hour = random.uniform(1.0, 4.0)
        base = datetime(2026, 6, 25, 0, 0, 0, tzinfo=timezone.utc) + timedelta(hours=hour)
    else:
        if random.random() < 0.6:
            hour = random.uniform(12.0, 22.0)
            base = datetime(2026, 6, 24, 0, 0, 0, tzinfo=timezone.utc) + timedelta(hours=hour)
        else:
            hour = random.uniform(5.0, 11.0)
            base = datetime(2026, 6, 25, 0, 0, 0, tzinfo=timezone.utc) + timedelta(hours=hour)
    return min(base, END - timedelta(minutes=margin_minutes))

def fraud_row(acc_id: str, amount: int, ts: datetime,
               is_home: bool = True, is_domestic: bool = True) -> dict:
    return {
        "account_id":  acc_id,
        "amount":      amount,
        "event_time":  min(ts, END - timedelta(seconds=1)).isoformat(),
        "is_home":     is_home,
        "is_domestic": is_domestic,
        "is_fraud":    True,
    }

In [43]:
f1_pool     = acc_avg[acc_avg < 3_000_000].index.tolist()
f1_accounts = random.sample(f1_pool, 4)

f1_locs = random.choices(
    ['home', 'other_domestic', 'foreign'],
    weights=[0.45, 0.30, 0.25],
    k=len(f1_accounts)
)

rows_f1 = []
for acc_id, loc in zip(f1_accounts, f1_locs):
    is_h = (loc == 'home')
    is_d = (loc != 'foreign')
    base_ts = fraud_base_time(night_prob=0.5, margin_minutes=10)
    for i in range(2):
        amount = round_vnd(random.uniform(800_000_000, 999_000_000))
        ts     = base_ts + timedelta(minutes=random.uniform(30, 90) * i)
        rows_f1.append(fraud_row(acc_id, amount, ts, is_home=is_h, is_domestic=is_d))

In [44]:
f2_pool    = acc_avg[(acc_avg >= 500_000) & (acc_avg <= 5_000_000)].index.tolist()
f2_accounts = random.sample(f2_pool, 260)

rows_f2 = []
for acc_id in f2_accounts:
    loc = random.choices(['home', 'other_domestic', 'foreign'],
                         weights=[0.45, 0.30, 0.25])[0]
    is_h = (loc == 'home')
    is_d = (loc != 'foreign')
    n_tx    = random.choice([5, 6, 6, 7])
    base_ts = fraud_base_time(night_prob=0.5, margin_minutes=n_tx * 43)
    times = [base_ts]
    for _ in range(n_tx - 1):
        times.append(times[-1] + timedelta(minutes=random.uniform(38, 43)))

    first_is_night = 1 <= base_ts.hour < 5

    for i, ts in enumerate(times):
        amount = round_vnd(random.uniform(100_000_000, 250_000_000))
        if i == 0 and not first_is_night:
            rows_f2.append({
                "account_id":  acc_id,
                "amount":      amount,
                "event_time":  min(ts, END - timedelta(seconds=1)).isoformat(),
                "is_home":     is_h,
                "is_domestic": is_d,
                "is_fraud":    False,
            })
        else:
            rows_f2.append(fraud_row(acc_id, amount, ts, is_home=is_h, is_domestic=is_d))

n_f2_fraud = sum(1 for r in rows_f2 if r["is_fraud"])
print(f"F2: {len(rows_f2)} total txs, {n_f2_fraud} fraud txs")

F2: 1574 total txs, 1439 fraud txs


In [45]:
f3_pool     = acc_avg[(acc_avg >= 300_000) & (acc_avg <= 2_000_000)].index.tolist()
f3_accounts = random.sample(f3_pool, 200)

rows_f3 = []
for acc_id in f3_accounts:
    loc = random.choices(['home', 'other_domestic', 'foreign'],
                         weights=[0.40, 0.35, 0.25])[0]
    is_h = (loc == 'home')
    is_d = (loc != 'foreign')
    n_tx       = random.randint(5, 10)
    base_ts    = fraud_base_time(night_prob=0.5, margin_minutes=60)
    window_min = random.uniform(50, 55)
    offsets = sorted([0.0] + [random.uniform(1, window_min) for _ in range(n_tx - 1)])

    first_is_night = 1 <= base_ts.hour < 5

    for i, off in enumerate(offsets):
        ts     = base_ts + timedelta(minutes=off)
        amount = round_vnd(random.uniform(80_000_000, 99_000_000))
        if i == 0 and not first_is_night:
            rows_f3.append({
                "account_id":  acc_id,
                "amount":      amount,
                "event_time":  min(ts, END - timedelta(seconds=1)).isoformat(),
                "is_home":     is_h,
                "is_domestic": is_d,
                "is_fraud":    False,
            })
        else:
            rows_f3.append(fraud_row(acc_id, amount, ts, is_home=is_h, is_domestic=is_d))

n_f3_fraud = sum(1 for r in rows_f3 if r["is_fraud"])
print(f"F3: {len(rows_f3)} total txs, {n_f3_fraud} fraud txs")

F3: 1514 total txs, 1409 fraud txs


In [46]:
selected = random.sample(all_normal_accounts, 230)
f4a_accounts = selected[:120]
f4b_accounts = selected[120:]

rows_f4 = []

for acc_id in f4a_accounts:
    base_avg  = float(acc_avg[acc_id])
    n_tx      = random.randint(3, 8)
    n_foreign = random.randint(2, max(2, n_tx - 1))

    is_domestic_seq = [False] * n_foreign + [True] * (n_tx - n_foreign)
    random.shuffle(is_domestic_seq)

    base_ts = fraud_base_time(night_prob=0.5, margin_minutes=n_tx * 85)
    times   = [base_ts]
    for _ in range(n_tx - 1):
        times.append(times[-1] + timedelta(minutes=random.uniform(65, 85)))

    for ts, is_dom in zip(times, is_domestic_seq):
        if is_dom:
            amount = round_vnd(np.random.normal(base_avg, base_avg * 0.15))
            rows_f4.append({
                "account_id": acc_id, "amount": amount,
                "event_time": min(ts, END - timedelta(seconds=1)).isoformat(),
                "is_home": True, "is_domestic": True, "is_fraud": False,
            })
        else:
            amount = round_vnd(np.random.normal(base_avg * 3.0, base_avg * 0.5))
            rows_f4.append({
                "account_id": acc_id, "amount": amount,
                "event_time": min(ts, END - timedelta(seconds=1)).isoformat(),
                "is_home": False, "is_domestic": False, "is_fraud": True,
            })

for acc_id in f4b_accounts:
    base_avg = float(acc_avg[acc_id])
    t0 = fraud_base_time(night_prob=0.5, margin_minutes=90)
    t1 = t0 + timedelta(minutes=random.uniform(25, 40))
    t2 = t1 + timedelta(minutes=random.uniform(25, 40))

    for ts, is_home in [(t0, True), (t1, False), (t2, True)]:
        if is_home:
            amount = round_vnd(np.random.normal(base_avg, base_avg * 0.15))
            rows_f4.append({
                "account_id": acc_id, "amount": amount,
                "event_time": min(ts, END - timedelta(seconds=1)).isoformat(),
                "is_home": True, "is_domestic": True, "is_fraud": False,
            })
        else:
            amount = round_vnd(np.random.normal(base_avg * 2.5, base_avg * 0.4))
            rows_f4.append({
                "account_id": acc_id, "amount": amount,
                "event_time": min(ts, END - timedelta(seconds=1)).isoformat(),
                "is_home": False, "is_domestic": True, "is_fraud": True,
            })

n_f4_fraud = sum(1 for r in rows_f4 if r["is_fraud"])
print(f"F4: {len(rows_f4)} total txs, {n_f4_fraud} fraud txs")

F4: 998 total txs, 500 fraud txs


In [47]:
f5_pool     = acc_avg[(acc_avg >= 500_000) & (acc_avg <= 4_000_000)].index.tolist()
f5_accounts = random.sample(f5_pool, 200)

rows_f5 = []
for acc_id in f5_accounts:
    loc = random.choices(['home', 'other_domestic', 'foreign'],
                         weights=[0.40, 0.30, 0.30])[0]
    is_h = (loc == 'home')
    is_d = (loc != 'foreign')

    n_tx    = random.randint(3, 6)
    span_s  = random.uniform(7_200, 18_000)  # 2–5 giờ
    base_ts = fraud_base_time(night_prob=0.5, margin_minutes=int(span_s / 60))
    times   = sorted(
        base_ts + timedelta(seconds=random.uniform(0, span_s))
        for _ in range(n_tx)
    )

    high_times = []
    for ts in times:
        if len(high_times) < 2:
            amount = round_vnd(random.uniform(100_000_000, 200_000_000))
            high_times.append(ts)
        else:
            if (ts - high_times[0]).total_seconds() > 3600:
                amount = round_vnd(random.uniform(100_000_000, 200_000_000))
                high_times.append(ts)
            else:
                amount = round_vnd(random.uniform(70_000_000, 99_000_000))
        if len(high_times) >= 3:
            high_times.pop(0)
        rows_f5.append(fraud_row(acc_id, amount, ts, is_home=is_h, is_domestic=is_d))

print(f"F5: {len(rows_f5)} fraud txs")

F5: 913 fraud txs


In [48]:
df_fraud = pd.DataFrame(rows_f1 + rows_f2 + rows_f3 + rows_f4 + rows_f5)

---

### Inject Anti Fraud Patterns

In [49]:
used_accounts = set(df_normal_transaction["account_id"].unique())
new_pool = [f"ML_{i:05d}" for i in range(100_000) if f"ML_{i:05d}" not in used_accounts]
random.shuffle(new_pool)

In [50]:
ptr = 0

rows_extra_normal = []

def normal_row(acc_id, amount, ts, is_home=True, is_domestic=True):
    return {
        "account_id":  acc_id,
        "amount":      max(50_000, amount),
        "event_time":  min(ts, END - timedelta(seconds=1)).isoformat(),
        "is_home":     is_home,
        "is_domestic": is_domestic,
        "is_fraud":    False,
    }

In [51]:
n1_acc = new_pool[ptr]; ptr += 3
t_n1   = fraud_base_time(night_prob=0.0, margin_minutes=120)
for lo, hi, offset_min in [(200_000_000, 300_000_000, 0), (900_000_000, 999_000_000, random.uniform(30, 90))]:
    rows_extra_normal.append(normal_row(n1_acc, round_vnd(random.uniform(lo, hi)),
                                          t_n1 + timedelta(minutes=offset_min)))

In [52]:
N2 = 65
for acc_id in new_pool[ptr:ptr + N2]:
    avg   = random.uniform(1_000_000, 5_000_000)
    t0    = fraud_base_time(night_prob=0.5, margin_minutes=120)
    n_tx  = random.randint(3, 7)
    times = sorted(t0 + timedelta(minutes=random.uniform(0, 120)) for _ in range(n_tx))
    for ts in times:
        amount = round_vnd(np.random.normal(avg, avg * 0.20))
        rows_extra_normal.append(normal_row(acc_id, amount, ts))
ptr += N2


In [53]:
N3 = 50
for acc_id in new_pool[ptr:ptr + N3]:
    avg     = random.uniform(1_000_000, 5_000_000)
    x       = random.randint(5, 7)
    n_home1 = random.randint(1, 2)
    n_away  = min(random.randint(2, 4), x - n_home1)
    n_home2 = x - n_home1 - n_away

    away_type = random.choice(['foreign', 'other_domestic'])
    is_d_away = (away_type == 'other_domestic') 

    t_base      = fraud_base_time(night_prob=0.3, margin_minutes=360)
    home1_times = sorted(t_base + timedelta(minutes=random.uniform(0, n_home1 * 15))
                         for _ in range(n_home1))

    away_start = home1_times[-1] + timedelta(minutes=random.uniform(61, 90))
    away_times = sorted(away_start + timedelta(minutes=random.uniform(0, n_away * 20))
                        for _ in range(n_away))

    home2_times = []
    if n_home2 > 0:
        home2_start = away_times[-1] + timedelta(minutes=random.uniform(61, 90))
        home2_times = sorted(home2_start + timedelta(minutes=random.uniform(0, n_home2 * 15))
                             for _ in range(n_home2))

    for ts in home1_times:
        rows_extra_normal.append(
            normal_row(acc_id, round_vnd(np.random.normal(avg, avg * 0.15)), ts))

    for ts in away_times:
        amount = round_vnd(random.uniform(avg * 1.0, avg * 3.0))
        rows_extra_normal.append(
            normal_row(acc_id, amount, ts, is_home=False, is_domestic=is_d_away))

    for ts in home2_times:
        rows_extra_normal.append(
            normal_row(acc_id, round_vnd(np.random.normal(avg, avg * 0.15)), ts))

ptr += N3


In [54]:
N4 = 65
for acc_id in new_pool[ptr:ptr + N4]:
    avg  = random.uniform(500_000, 3_000_000)
    hour = random.uniform(1.0, 4.0)
    t0   = datetime(2026, 6, 25, 0, 0, 0, tzinfo=timezone.utc) + timedelta(hours=hour)
    t1   = t0 + timedelta(minutes=random.uniform(20, 50))
    for ts in [t0, t1]:
        amount = round_vnd(np.random.normal(avg, avg * 0.20))
        rows_extra_normal.append(normal_row(acc_id, amount, ts))
ptr += N4

In [55]:
N5 = 50
for acc_id in new_pool[ptr:ptr + N5]:
    n_tx    = random.randint(5, 7)
    base_ts = fraud_base_time(night_prob=0.5, margin_minutes=120)
    times   = sorted(base_ts + timedelta(minutes=random.uniform(0, 120)) for _ in range(n_tx))
    for ts in times:
        amount = round_vnd(random.uniform(30_000_000, 100_000_000))
        rows_extra_normal.append(normal_row(acc_id, amount, ts))
ptr += N5

In [56]:
print(f"Normal extra: {len(rows_extra_normal)} txs  (N1=2, N2={N2*3}, N3={N3*3}, N4={N4*2})")

Normal extra: 1053 txs  (N1=2, N2=195, N3=150, N4=130)


In [57]:
df_extra = pd.DataFrame(rows_extra_normal)

---

### Write DataFrame to CSV

In [58]:
df_final = (
    pd.concat([df_normal_transaction, df_fraud, df_extra], ignore_index=True)
      .sort_values("event_time")
      .reset_index(drop=True)
)

In [59]:
n_fraud = int(df_final["is_fraud"].sum())
n_total = len(df_final)
print(f"Total : {n_total:,}")
print(f"Fraud : {n_fraud:,}  ({n_fraud/n_total*100:.2f}%)")
print(f"Normal: {n_total - n_fraud:,}")
df_final.head()

Total : 206,064
Fraud : 4,269  (2.07%)
Normal: 201,795


,account_id,amount,event_time,is_home,is_domestic,is_fraud
0,ML_81908,3116000,2026-06-24T12:00:00.605527+00:00,True,True,False
1,ML_70721,7763000,2026-06-24T12:00:00.719117+00:00,True,True,False
2,ML_33203,951000,2026-06-24T12:00:01.057246+00:00,True,True,False
3,ML_43504,2553000,2026-06-24T12:00:01.433330+00:00,True,True,False
4,ML_08929,3880000,2026-06-24T12:00:01.487840+00:00,True,True,False


In [60]:
df_final.to_csv("../data/data_1.csv", index=False)
print(f"Saved: ../data/data_1.csv  ({len(df_final):,} rows)")

Saved: ../data/data_1.csv  (206,064 rows)


---